# Лабораторная работа №8 по ТМО
## Перфильев Виктор Анатольевич
## Группа ИБМ3-65Б

#### Цель работы. Изучение возможностей демонстрации моделей машинного обучения с помощью веб-приложений.

## 1. Импорт библиотек и загрузка датасета

In [10]:
import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, explained_variance_score, mean_absolute_percentage_error)
import warnings
warnings.filterwarnings('ignore')

housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target
feature_names = housing.feature_names
target_name = housing.target_names[0]

## 2. Разделение на обучающую и тестовую выборки

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Обучение и визуализация

#### Обучает GradientBoostingRegressor с заданными гиперпараметрами и возвращает:
####     - график: предсказания vs истинные значения
####     - график: важность признаков
####     - DataFrame с метриками
####     - DataFrame с первыми 10 предсказаниями
####     - параметры модели

In [17]:
def train_model(n_estimators, learning_rate, max_depth, min_samples_split,
                min_samples_leaf, subsample, loss, criterion):
    
    # Обучение модели
    model = GradientBoostingRegressor(
        n_estimators=int(n_estimators),
        learning_rate=learning_rate,
        max_depth=int(max_depth),
        min_samples_split=int(min_samples_split),
        min_samples_leaf=int(min_samples_leaf),
        subsample=subsample,
        loss=loss,
        criterion=criterion,
        random_state=42
    )
    model.fit(X_train, y_train)
    
    # Предсказания
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Метрики на тестовой выборке
    metrics = {
        "MAE (Mean Absolute Error)": mean_absolute_error(y_test, y_pred_test),
        "MSE (Mean Squared Error)": mean_squared_error(y_test, y_pred_test),
        "RMSE (Root Mean Squared Error)": np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "R² (R-squared)": r2_score(y_test, y_pred_test),
        "Explained Variance": explained_variance_score(y_test, y_pred_test),
        "MAPE (Mean Absolute Percentage Error)": mean_absolute_percentage_error(y_test, y_pred_test)
    }
    
    # DataFrame метрик
    metrics_df = pd.DataFrame(list(metrics.items()), columns=['Metric', 'Value'])
    metrics_df['Value'] = metrics_df['Value'].round(4)
    
    # DataFrame предсказаний (первые 10)
    predictions_df = pd.DataFrame({
        'Actual': y_test[:10].values,
        'Predicted': y_pred_test[:10].round(4),
        'Error': (y_test[:10].values - y_pred_test[:10]).round(4)
    })
    
    # График 1: Предсказания vs Истинные значения
    fig_scatter, ax_scatter = plt.subplots(figsize=(8, 6))
    ax_scatter.scatter(y_test, y_pred_test, alpha=0.6, edgecolors='k', linewidth=0.5)
    ax_scatter.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Идеальная линия')
    ax_scatter.set_xlabel('Истинные значения')
    ax_scatter.set_ylabel('Предсказанные значения')
    ax_scatter.set_title(f'Предсказания модели (R² = {metrics["R² (R-squared)"]:.4f})')
    ax_scatter.legend()
    ax_scatter.grid(True, alpha=0.3)
    
    # График 2: Важность признаков
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    fig_imp, ax_imp = plt.subplots(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, len(feature_names)))
    ax_imp.barh(range(len(feature_names)), importances[indices], color=colors, align='center')
    ax_imp.set_yticks(range(len(feature_names)))
    ax_imp.set_yticklabels([feature_names[i] for i in indices])
    ax_imp.set_xlabel('Важность признака')
    ax_imp.set_title('Важность признаков (Feature Importances)')
    ax_imp.invert_yaxis()
    ax_imp.grid(True, alpha=0.3)
    
    # Информация о гиперпараметрах
    params_info = {
        "n_estimators": model.n_estimators,
        "learning_rate": model.learning_rate,
        "max_depth": model.max_depth,
        "min_samples_split": model.min_samples_split,
        "min_samples_leaf": model.min_samples_leaf,
        "subsample": model.subsample,
        "loss": model.loss,
        "criterion": model.criterion,
        "Train score (R²)": round(model.score(X_train, y_train), 4),
        "Test score (R²)": round(metrics["R² (R-squared)"], 4)
    }
    params_df = pd.DataFrame(list(params_info.items()), columns=['Parameter', 'Value'])
    params_df['Value'] = params_df['Value'].astype(str)
    
    return fig_scatter, fig_imp, metrics_df, predictions_df, params_df

## 4. Интерфейс Gradio 

In [22]:
with gr.Blocks(title="Gradient Boosting Regressor - California Housing", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    #Gradient Boosting Regressor — предсказание цен на жильё в Калифорнии
    
    **Датасет:** California Housing (20,640 samples, 8 features)  
    **Целевая переменная:** MedHouseVal — медианная стоимость дома в районе (в сотнях тысяч долларов)
    
    Настройте гиперпараметры модели ниже и нажмите **Обучить модель**.
    """)
    
    with gr.Row():
        # Левая колонка: гиперпараметры
        with gr.Column(scale=1):
            gr.Markdown("### Основные параметры")
            n_estimators = gr.Slider(10, 300, value=100, step=10, label="n_estimators (число деревьев)")
            learning_rate = gr.Slider(0.01, 1.0, value=0.1, step=0.01, label=" learning_rate (скорость обучения)")
            max_depth = gr.Slider(1, 15, value=3, step=1, label="max_depth (максимальная глубина)")
            
            gr.Markdown("### Дополнительные параметры")
            min_samples_split = gr.Slider(2, 20, value=2, step=1, label="min_samples_split (мин. объектов для разбиения)")
            min_samples_leaf = gr.Slider(1, 20, value=1, step=1, label="min_samples_leaf (мин. объектов в листе)")
            subsample = gr.Slider(0.5, 1.0, value=1.0, step=0.05, label="subsample (доля выборки для каждого дерева)")
            loss = gr.Dropdown(['squared_error', 'absolute_error', 'huber', 'quantile'], value='squared_error', label="loss (функция потерь)")
            criterion = gr.Dropdown(['friedman_mse', 'squared_error'], value='friedman_mse', label="criterion (критерий разбиения)")
            
            train_btn = gr.Button("🚀 Обучить модель", variant="primary")
        
        # Правая колонка: результаты
        with gr.Column(scale=2):
            gr.Markdown("### Визуализация результатов")
            with gr.Tabs():
                with gr.TabItem("Предсказания vs Истина"):
                    plot_scatter = gr.Plot(label="Диаграмма рассеяния")
                with gr.TabItem("Важность признаков"):
                    plot_importance = gr.Plot(label="Feature Importances")
                with gr.TabItem("Метрики"):
                    metrics_table = gr.Dataframe(label="Метрики качества", interactive=False)
                with gr.TabItem("Примеры предсказаний"):
                    predictions_table = gr.Dataframe(label="Первые 10 значений тестовой выборки", interactive=False)
                with gr.TabItem("Параметры модели"):
                    params_table = gr.Dataframe(label="Использованные гиперпараметры", interactive=False)
    
    # Обработчик события
    train_btn.click(
        fn=train_model,
        inputs=[n_estimators, learning_rate, max_depth, min_samples_split,
                min_samples_leaf, subsample, loss, criterion],
        outputs=[plot_scatter, plot_importance, metrics_table, predictions_table, params_table]
    )

## 5. Запуск приложения

In [23]:
demo.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
